# D1 MRI interpolation sensitivity: a short matched training study

Compare **0: nearest, 1: linear, 2: quadratic spline, 3: cubic spline** on the
normalized TBI mouse dataset. These are SciPy spline orders, not SimpleITK enums.
The selected order applies to MRI grid resampling in training, validation and
full-volume inference. Masks always use nearest neighbour. Physical affine
augmentation stays linear in every run, isolating the grid-resampling choice.

The pilot retains the original **80³ patches and (16, 32, 64, 128, 256) widths**,
and trains four fresh models with identical animals, seeds, evaluation crop
coordinates, augmentation policy, loss and postprocessing. No pretrained weights
carry an earlier interpolation convention into this comparison.

Defaults: **10 epochs, 4 training batches per epoch, one seed**, using all D1
animals with a shared fixed split. Training is disabled until `RUN_STUDY=True`.
Validation/test batches are complete; only the training budget is shortened.
This is a screening study, not evidence that every model has converged. Use
`STEPS_PER_EPOCH=None` for complete training epochs and a fresh study for a larger
budget. Additional seeds are configurable. Fixed seeds do not guarantee bitwise
identical GPU execution or augmentation scheduling.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, replace
import sys
import json

REPO_ROOT: Path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/models/networks.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import nibabel as nib
import numpy as np
import pandas as pd
import tensorflow as tf
import plotly.express as px
from IPython.display import display

from src.experiments.config import (
    DataConfig, DatasetSpec, EvaluationPatchConfig, ExperimentConfig, FixedSplitConfig,
    InferenceConfig, InitializationConfig, LabelConfig, LossConfig, ModelConfig,
    PostprocessingConfig, PreprocessingConfig,
)
from src.experiments.data import prepare_data, make_preprocessor
from src.experiments.ablation import study_selection, run_study
from src.experiments.training import configure_runtime
from src.evaluation.k_fold import TrainingConfig
from src.preprocessing.preprocessor import Preprocessor, Resample

REFERENCE: Path = REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49" / "Anat" / "TBI_fm_19_49_N4.nii.gz"
ref_img = nib.load(REFERENCE)
print("Reference axes:", nib.aff2axcodes(ref_img.affine))
runtime = configure_runtime(seed=42, memory_growth=True, mixed_precision=False)

## 1. Visible training budget and common configuration

Order 1 is the reference. N4 is already present in the input files. Intensity
blur/noise occur on the native grid, as in notebook 01; affine augmentation,
centre-biased random cropping (`mode="center", std=None`) and LR flipping remain
shared. Training merges the lateral ventricles; full-volume evaluation splits
them by centroid and restores original IDs.

An existing `SPLIT_FILE` can enforce exact memberships. The default comes from the selected
reference fold and is saved with the study. Do not reuse a final test cohort
for repeated tuning: use validation for choosing an interpolation setting. The
test comparison describes these four predeclared candidates.

The default `REFERENCE_SPLITS` is `DATA_ROOT.parent / "D1_splits_final.json"`.
Historical `val` groups become independent outer test sets; inner validation is
sampled only from each saved training pool, using the shared seed and fraction.
Fixed runs use `REFERENCE_FOLD=0`; K-Fold runs use the complete reference plan.
MouseFull adds anatomy-balanced SHAM folds without changing any D1 role.
Explicit `SPLIT_FILE`/`FOLD_FILE` and resumed memberships take precedence.
Set `REFERENCE_SPLITS=None` to generate a fresh plan. Mouse experiments start
from scratch on these memberships; weights are not shared across folds.


In [ ]:
DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
REFERENCE_SPLITS: Path | None = DATA_ROOT.parent / "D1_splits_final.json"
REFERENCE_FOLD: int = 0  # Fixed studies reuse this fold; CV uses all reference folds.
STUDY_DIR: Path = REPO_ROOT / "results" / "interpolation" / f"d1_orders_0_1_2_3_reference_fold_{REFERENCE_FOLD}"
SPLIT_FILE: Path | None = None
RUN_STUDY: bool = False
RESUME_STUDY: bool = False

ORDERS: tuple[int, ...] = (0, 1, 2, 3)
TRAINING_SEEDS: tuple[int, ...] = (42,)
EPOCHS: int = 10
STEPS_PER_EPOCH: int | None = 4
BATCH_SIZE: int = 8
DATA_MULTIPLIER: int = 1
PATCH_SIZE: tuple[int, int, int] = (80, 80, 80)
FILTERS: tuple[int, int, int, int, int] = (16, 32, 64, 128, 256)
INFERENCE_STRIDE: int = 40
SAVE_PREDICTIONS: bool = False

training = TrainingConfig(
    model_name="d1_interpolation", epochs=EPOCHS, batch_size=BATCH_SIZE,
    steps_per_epoch=STEPS_PER_EPOCH, data_multiplier=DATA_MULTIPLIER,
    seed=42, output_dir=str(REPO_ROOT / "results"),
    monitor="val_loss", monitor_mode="min", default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=PATCH_SIZE, spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    resampling_order=1, intensity_augmentation_grid="native",
    flip_probability=0.5,
)
postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=True,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference = InferenceConfig(
    enabled=True, stride=INFERENCE_STRIDE, save_predictions=SAVE_PREDICTIONS,
    original_label_ids=True, postprocessing=postprocessing,
)
config = ExperimentConfig(
    data=DataConfig(root=DATA_ROOT, datasets=(DatasetSpec(name="D1"),)),
    preprocessing=preprocessing, training=training,
    labels=LabelConfig(unknown="background"),
    model=ModelConfig(
        architecture="3drnet", filters=FILTERS,
        residual=True, attention=True, max_pooling=True, classifier=False,
    ),
    loss=LossConfig(regions="dice_ce", gamma=0.8, brain_alpha=0.6, batch_wise=True),
    inference=inference, initialization=InitializationConfig(),
    fixed_split=FixedSplitConfig(validation_fraction=0.2, test_fraction=0.1),
    evaluation_patches=EvaluationPatchConfig(
        patches_per_scan=3, sampling="center_biased", seed=42, left_right_pairs=False,
    ),
    learning_rate=0.001, split_seed=42,
)
config.validate()

## 2. Configure callbacks directly

The pilot uses a fixed learning rate and epoch budget so every model receives
the same number of optimizer steps. Logging callbacks are visible below. Add
adaptive callbacks here only as an explicit shared protocol change in a fresh
study. Best validation weights and recovery checkpoints are saved by the core;
each fit receives fresh callback instances.

In [ ]:
callbacks = [
    tf.keras.callbacks.TensorBoard(log_dir="logs", histogram_freq=0),
    tf.keras.callbacks.CSVLogger(filename="training.csv"),
]

In [ ]:
prepared = prepare_data(config)
selection = study_selection(
    data=prepared, mode="fixed_split", study_dir=STUDY_DIR,
    split_file=SPLIT_FILE, resume=RESUME_STUDY,
    reference_file=REFERENCE_SPLITS, reference_fold=REFERENCE_FOLD,
)
split = selection.split
display(selection.summary)
display(prepared.annotation_summary())
print("Study:", STUDY_DIR)
print("Fits:", len(ORDERS) * len(TRAINING_SEEDS))

## 3. Check which inputs actually require resampling

Reorientation and unit correction precede resampling. Scans already at the
target spacing bypass resampling, so changing spline order cannot alter their
input at this step. The audit below uses the actual production prefix and
reports this by split and sequence. Inspect T2w separately instead of assuming
every D1 scan is affected. The training cell refuses a completely unaffected
inventory; no artificial displacement or resolution change is introduced.
The flag follows the resampler's exact tolerance: tiny header rounding can mark
nominally isotropic scans as affected. Inspect the actual spacing and relative
change rather than treating every flagged scan as a large resolution change.

In [ ]:
base_pipeline = make_preprocessor(config=config, stage="inference")
resample_index = next(index for index, step in enumerate(base_pipeline.steps) if isinstance(step, Resample))
prefix_steps = base_pipeline.steps[:resample_index]
prefix = Preprocessor(steps=prefix_steps)

roles: dict[str, str] = {}
for role, animals in (("train", split.train), ("validation", split.validation), ("test", split.test)):
    for animal in animals:
        roles[animal] = role

audit_rows: list[dict[str, object]] = []
for key in prepared.loader.get_subjects_list():
    scan = prepared.loader.get_subject(key)
    subject = prepared.subjects.subject_for_scan(key)
    corrected = prefix.preprocess(imgs=scan.image)
    spacing = nib.affines.voxel_sizes(corrected.affine)
    needs_resampling = not np.allclose(spacing, config.preprocessing.spacing_mm, rtol=0, atol=1e-8)
    target_spacing = np.asarray(config.preprocessing.spacing_mm)
    relative_change = np.abs(spacing / target_spacing - 1)
    maximum_change = float(relative_change.max())
    anisotropy = float(spacing.max() / spacing.min())
    modality = scan.metadata.get("modality", scan.modality)
    audit_rows.append({
        "Subject": subject, "Scan": scan.id, "Dataset": scan.subject_group[0],
        "DatasetRole": roles[subject], "Modality": modality,
        "SpacingBeforeResampling": tuple(float(value) for value in spacing),
        "MaxRelativeSpacingChange": maximum_change, "SpacingAnisotropy": anisotropy,
        "ResamplingNeeded": needs_resampling,
    })

grid_audit = pd.DataFrame(audit_rows)
grid_counts = grid_audit.groupby(["DatasetRole", "Modality", "ResamplingNeeded"]).agg(
    Animals=("Subject", "nunique"), Scans=("Scan", "size"),
).reset_index()
display(grid_counts)
display(grid_audit)

In [ ]:
@dataclass(frozen=True)
class InterpolationVariant:
    """One MRI resampling order; every other experiment setting is shared."""

    name: str
    order: int

    def configuration(self, base: ExperimentConfig) -> ExperimentConfig:
        """Store the order in the normal experiment/recovery configuration."""
        settings = replace(base.preprocessing, resampling_order=self.order)
        configured = replace(base, preprocessing=settings)
        configured.validate()
        return configured


if len(set(ORDERS)) != len(ORDERS) or 1 not in ORDERS:
    raise ValueError("Choose distinct interpolation orders and retain order 1 as reference")

variants: list[InterpolationVariant] = []
plan_rows: list[dict[str, object]] = []
for order in ORDERS:
    name = f"order_{order}"
    variant = InterpolationVariant(name=name, order=order)
    variant.configuration(config)
    variants.append(variant)
    plan_rows.append({
        "Variant": name, "MRIResamplingOrder": order,
        "MaskOrder": 0, "AffineMRI": "linear", "Epochs": EPOCHS,
        "StepsPerEpoch": STEPS_PER_EPOCH, "Initialization": "scratch",
    })

plan = pd.DataFrame(plan_rows)
display(plan)

## 4. Run or resume the four models

Set `RUN_STUDY=True` in configuration and run the following cell. To resume an
interruption, keep the same study directory/settings, set `RESUME_STUDY=True`
and rerun the preceding cells. Completed fits are reused. Changing orders,
epochs, steps or seeds requires a fresh study directory.

The same saved evaluation coordinates are reused, with each model's selected
MRI order and identical nearest-neighbour GT. Validation selects best weights;
full-volume validation/test metrics are then measured once per evaluation pass.
No training starts with the default switch.

In [ ]:
study = None
if RUN_STUDY:
    if not grid_audit["ResamplingNeeded"].any():
        raise ValueError("No input requires target resampling; these orders would produce identical inputs")
    train_generator = prepared.make_generator(split.train, training=True)
    if STEPS_PER_EPOCH is not None and STEPS_PER_EPOCH > len(train_generator):
        raise ValueError("STEPS_PER_EPOCH exceeds the available training batches; reduce it or use None")
    del train_generator

    study = run_study(
        data=prepared, variants=variants, plan=plan, study_dir=STUDY_DIR,
        mode="fixed_split", split=split, seeds=TRAINING_SEEDS,
        resume=RESUME_STUDY, callbacks=callbacks,
    )
    grid_audit.to_csv(STUDY_DIR / "resampling_audit.csv", index=False)
    print("Study saved to:", study.study_dir)
else:
    print("Review the configuration and grid audit, then set RUN_STUDY=True.")

In [ ]:
CURVE_METRIC: str = "loss"  # Also try regions_mean_dice after inspecting available metrics.

history_rows: list[dict[str, object]] = []
if study is not None:
    for run in study.runs.values():
        variant_name = run.run_dir.parent.name
        saved_config = ExperimentConfig.load(run.run_dir / "experiment.json")
        for fold, history in enumerate(run.histories):
            for metric, values in history.items():
                for epoch, value in enumerate(values, start=1):
                    if value is not None:
                        history_rows.append({
                            "Variant": variant_name, "Seed": str(saved_config.training.seed),
                            "Fold": fold, "Epoch": epoch, "Metric": metric, "Value": value,
                        })
    histories = pd.DataFrame(history_rows)
    print("Available metrics:", sorted(histories["Metric"].unique()))
    curve_names = [CURVE_METRIC, f"val_{CURVE_METRIC}"]
    curve_rows = histories.loc[histories["Metric"].isin(curve_names)]
    if not curve_rows.empty:
        curve_figure = px.line(
            curve_rows, x="Epoch", y="Value", color="Variant", line_dash="Seed",
            facet_row="Metric", title="Matched short-budget training curves",
        )
        curve_figure.show(renderer="vscode")

## 5. Compare validation and test measurements

This section reads saved results, so it also works after a restart with the
setup/configuration cells and an explicit `RESULTS_DIR`; no fitting is required.
Validation lesion Dice supplies the exploratory ranking. Full-volume reports
include postprocessed lesion, separate ventricles and brain metrics: Dice, IoU,
HD95, ASSD, sensitivity, precision and volume errors. Native deterministic patch
metrics remain separate. Paired test comparisons use **order 1 as reference**;
their differences are candidate minus linear, with animal intervals and Holm
correction. Inspect sample counts and empty-mask statuses.

One split and one seed do not establish a universally better interpolation
method. A short-budget difference may reflect convergence speed. Do not select
settings from the test ranking and then describe that cohort as untouched.

In [ ]:
from src.evaluation.paper_tables import PaperTableConfig, paper_tables_from_folder

RESULTS_DIR: Path = STUDY_DIR
BOOTSTRAP_SAMPLES: int = 1000

validation_tables = None
test_tables = None
metric_file = RESULTS_DIR / "full_volume_metrics.csv"
if metric_file.is_file() and metric_file.stat().st_size > 10:
    validation_settings = PaperTableConfig(
        roles=("validation",), reference_variant="order_1", bootstrap_samples=BOOTSTRAP_SAMPLES,
    )
    test_settings = PaperTableConfig(
        roles=("test",), reference_variant="order_1", bootstrap_samples=BOOTSTRAP_SAMPLES,
    )
    validation_tables = paper_tables_from_folder(RESULTS_DIR, settings=validation_settings)
    test_tables = paper_tables_from_folder(RESULTS_DIR, settings=test_settings)

    validation_scores = validation_tables.statistics
    lesion_dice = validation_scores.loc[
        validation_scores["Head"].eq("regions") & validation_scores["Region"].eq("Lesion")
        & validation_scores["Metric"].eq("Dice")
    ]
    ranking_columns = ["Variant", "Seed", "Mean", "Animals", "AnimalCILower", "AnimalCIUpper"]
    validation_ranking = lesion_dice[ranking_columns].sort_values(["Seed", "Mean"], ascending=[True, False])
    print("Exploratory validation lesion-Dice ranking:")
    display(validation_ranking)
    print("Native validation patch measurements:")
    display(validation_tables.patches)
    print("Held-out test metrics and anatomy/brain averages:")
    display(test_tables.fold_coverage)
    display(test_tables.segmentation)
    display(test_tables.region_sets)
    display(test_tables.animal_intervals)
    display(test_tables.patches)
    display(test_tables.comparisons)
    display(test_tables.costs)
else:
    print("No saved study measurements yet. Run the study or select an explicit RESULTS_DIR.")

## 6. Separate T1w/T2w and affected/unaffected scans

The tables below average scans within animals and keep seeds separate. They
show Dice by MRI sequence and whether target resampling actually occurred.
Inspect the individual points and counts: a pooled D1 average can dilute an
effect confined to anisotropic T2w inputs. Empty-reference Dice conventions
remain visible in the saved full-volume CSV, alongside boundary statuses.

In [ ]:
if test_tables is not None:
    measurements = pd.read_csv(metric_file)
    audit_file = RESULTS_DIR / "resampling_audit.csv"
    if not audit_file.is_file():
        raise FileNotFoundError("Select the study's saved resampling_audit.csv for sequence-specific interpretation")
    saved_audit = pd.read_csv(audit_file)
    join_keys = ["Subject", "Scan", "Dataset"]
    audit_columns = [*join_keys, "ResamplingNeeded"]
    measurements = measurements.merge(saved_audit[audit_columns], on=join_keys, how="left", validate="many_to_one")
    if measurements["ResamplingNeeded"].isna().any():
        raise ValueError("Some measured scans are missing from the saved grid audit")

    test_dice = measurements.loc[measurements["DatasetRole"].eq("test") & measurements["Metric"].eq("Dice")].copy()
    grouping = ["Variant", "Seed", "Modality", "ResamplingNeeded", "Head", "Region"]
    animal_dice = test_dice.groupby([*grouping, "Subject"], dropna=False)["Value"].mean().reset_index()
    sequence_summary = animal_dice.groupby(grouping, dropna=False).agg(
        Animals=("Value", "count"), MeanDice=("Value", "mean"), SDDice=("Value", "std"),
    ).reset_index()
    display(sequence_summary)
    sequence_figure = px.box(
        animal_dice, x="Variant", y="Value", color="Modality", facet_col="Region",
        facet_col_wrap=3, points="all", hover_data=["Subject", "Seed", "ResamplingNeeded"],
        title="Test animal Dice by sequence (descriptive; empty-mask conventions retained)",
        labels={"Value": "Animal Dice"},
    )
    sequence_figure.show(renderer="vscode")

In [ ]:
EXPORT_PAPER_TABLES: bool = False
PAPER_OUTPUT: Path | None = None  # Fresh explicit folder; never overwrite an existing export.

if EXPORT_PAPER_TABLES:
    if validation_tables is None or test_tables is None or PAPER_OUTPUT is None:
        raise ValueError("Load the results and select a fresh PAPER_OUTPUT folder")
    PAPER_OUTPUT.mkdir(parents=True, exist_ok=False)
    validation_output = PAPER_OUTPUT / "validation"
    test_output = PAPER_OUTPUT / "test"
    validation_tables.export(validation_output)
    test_tables.export(test_output)
    validation_ranking.to_csv(PAPER_OUTPUT / "validation_lesion_dice_ranking.csv", index=False)
    sequence_summary.to_csv(PAPER_OUTPUT / "test_sequence_summary.csv", index=False)
    print("Paper tables saved to:", PAPER_OUTPUT)